# CartPole con DQN — laboratorio compacto


In [1]:
%pip install -q "gymnasium[classic-control]"

In [2]:
import random
from collections import deque, namedtuple

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation

import torch
import torch.nn as nn
import torch.optim as optim

from IPython.display import HTML, display


CONFIG = {

    # DQN
    "gamma": 1.0,
    "lr": 1e-3,

    # LR decay lineal opcional
    "lr_decay": False,
    "lr_end": 1e-4,
    "lr_decay_steps": 30_000,

    # Replay buffer
    "buffer_size": 3_000,
    "min_buffer_size": 1_000,
    "batch_size": 64,

    # Updates
    "train_every": 1,
    "target_update_every": 100,

    # Epsilon-greedy
    "epsilon_start": 1.0,
    "epsilon_end": 0.05,
    "epsilon_decay_steps": 30_000,

    # Red
    "hidden_layers": [128, 128, 128],

    # Reward:
    # "step" -> +1 por step
    # "terminal_length" -> 0, ..., 0, T
    "reward_mode": "step",

    # NO TOCAR
    "seed": 42,
    "episodes": 500,
    "max_steps": 1000,
    "eval_every": 100,
    "eval_episodes": 30,
    "final_eval_episodes": 300,
    "render_eval": True,
    "frame_skip": 2,
}

## DQN

In [3]:
Transition = namedtuple(
    "Transition",
    ["state", "action", "reward", "next_state", "done"]
)


class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def add(self, *transition):
        self.buffer.append(Transition(*transition))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return Transition(*zip(*batch))

    def __len__(self):
        return len(self.buffer)


class QNetwork(nn.Module):
    def __init__(self, obs_dim, n_actions, hidden_layers):
        super().__init__()

        layers = []
        in_dim = obs_dim

        for hidden_dim in hidden_layers:
            layers += [nn.Linear(in_dim, hidden_dim), nn.GELU()]
            in_dim = hidden_dim

        layers.append(nn.Linear(in_dim, n_actions))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [4]:
def linear_decay(start, end, step, decay_steps):
    p = min(step / decay_steps, 1.0)
    return start + p * (end - start)


def epsilon_at(step, cfg):
    return linear_decay(
        cfg["epsilon_start"],
        cfg["epsilon_end"],
        step,
        cfg["epsilon_decay_steps"],
    )


def lr_at(step, cfg):
    if not cfg["lr_decay"]:
        return cfg["lr"]

    return linear_decay(
        cfg["lr"],
        cfg["lr_end"],
        step,
        cfg["lr_decay_steps"],
    )


def reward_at(mode, episode_length, done):
    if mode == "step":
        return 1.0

    if mode == "terminal_length":
        return float(episode_length) if done else 0.0

    raise ValueError("reward_mode inválido")

In [5]:
def dqn_update(q_net, target_net, optimizer, buffer, cfg, device):
    batch = buffer.sample(cfg["batch_size"])

    states = torch.as_tensor(
        np.array(batch.state), dtype=torch.float32, device=device
    )
    actions = torch.as_tensor(
        batch.action, dtype=torch.long, device=device
    )
    rewards = torch.as_tensor(
        batch.reward, dtype=torch.float32, device=device
    )
    next_states = torch.as_tensor(
        np.array(batch.next_state), dtype=torch.float32, device=device
    )
    dones = torch.as_tensor(
        batch.done, dtype=torch.float32, device=device
    )

    q_sa = q_net(states).gather(1, actions[:, None]).squeeze(1)

    with torch.no_grad():
        next_q = target_net(next_states).max(dim=1).values
        target = rewards + cfg["gamma"] * (1 - dones) * next_q

    loss = nn.functional.smooth_l1_loss(q_sa, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

## Evaluación greedy y render

In [6]:
def greedy_episode(model, cfg, seed, render=False):
    device = next(model.parameters()).device

    env = gym.make(
        "CartPole-v1",
        render_mode="rgb_array" if render else None
    )

    state, _ = env.reset(seed=seed)

    length = 0
    frames = []

    if render:
        frames.append(env.render())

    model.eval()

    for _ in range(cfg["max_steps"]):
        with torch.no_grad():
            state_t = torch.as_tensor(
                state, dtype=torch.float32, device=device
            ).unsqueeze(0)

            action = model(state_t).argmax(dim=1).item()

        state, _, terminated, truncated, _ = env.step(action)
        length += 1

        if render:
            frames.append(env.render())

        if terminated or truncated:
            break

    env.close()
    model.train()

    return length, frames


def evaluate(model, cfg, n_episodes, seed_offset):
    lengths = [
        greedy_episode(
            model,
            cfg,
            cfg["seed"] + seed_offset + i,
            render=False
        )[0]
        for i in range(n_episodes)
    ]

    return np.array(lengths)


def show_episode(frames, frame_skip=2, fps=25):
    frames = frames[::frame_skip]

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.axis("off")

    image = ax.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=1000 / fps,
        blit=True
    )

    plt.close(fig)
    display(HTML(anim.to_jshtml()))

## Plot

In [7]:
def plot_progress(history, eval_history=None):
    plt.figure(figsize=(10, 4))

    plt.plot(
        history["episode"],
        history["length"],
        label="Train"
    )

    if eval_history and len(eval_history["episode"]) > 0:
        plt.plot(
            eval_history["episode"],
            eval_history["mean_length"],
            marker="o",
            linewidth=2,
            label="Greedy eval"
        )

    plt.xlabel("Episode")
    plt.ylabel("Episode length")
    plt.legend()
    plt.show()

## Entrenamiento

In [8]:
def train(cfg):
    random.seed(cfg["seed"])
    np.random.seed(cfg["seed"])
    torch.manual_seed(cfg["seed"])

    device = torch.device(
        "cuda" if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available()
        else "cpu"
    )

    env = gym.make("CartPole-v1")

    obs_dim = env.observation_space.shape[0]
    n_actions = env.action_space.n

    q_net = QNetwork(
        obs_dim,
        n_actions,
        cfg["hidden_layers"]
    ).to(device)

    target_net = QNetwork(
        obs_dim,
        n_actions,
        cfg["hidden_layers"]
    ).to(device)

    target_net.load_state_dict(q_net.state_dict())

    optimizer = optim.Adam(
        q_net.parameters(),
        lr=cfg["lr"]
    )

    buffer = ReplayBuffer(cfg["buffer_size"])

    history = {
        "episode": [],
        "length": [],
    }

    eval_history = {
        "episode": [],
        "mean_length": [],
    }

    global_step = 0
    update_step = 0

    for episode in range(1, cfg["episodes"] + 1):
        state, _ = env.reset(seed=cfg["seed"] + episode)

        episode_length = 0

        for _ in range(cfg["max_steps"]):
            epsilon = epsilon_at(global_step, cfg)

            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    s = torch.as_tensor(
                        state,
                        dtype=torch.float32,
                        device=device
                    ).unsqueeze(0)

                    action = q_net(s).argmax(dim=1).item()

            next_state, _, terminated, truncated, _ = env.step(action)

            episode_length += 1
            done = terminated or truncated

            reward = reward_at(
                cfg["reward_mode"],
                episode_length,
                done
            )

            buffer.add(
                state,
                action,
                reward,
                next_state,
                done
            )

            state = next_state
            global_step += 1

            if (
                len(buffer) >= cfg["min_buffer_size"]
                and global_step % cfg["train_every"] == 0
            ):
                current_lr = lr_at(global_step, cfg)

                for group in optimizer.param_groups:
                    group["lr"] = current_lr

                dqn_update(
                    q_net,
                    target_net,
                    optimizer,
                    buffer,
                    cfg,
                    device
                )

                update_step += 1

                if update_step % cfg["target_update_every"] == 0:
                    target_net.load_state_dict(
                        q_net.state_dict()
                    )

            if done:
                break

        history["episode"].append(episode)
        history["length"].append(episode_length)

        if (
            episode % cfg["eval_every"] == 0
            or episode == cfg["episodes"]
        ):
            lengths = evaluate(
                q_net,
                cfg,
                cfg["eval_episodes"],
                seed_offset=100_000
            )

            eval_history["episode"].append(episode)
            eval_history["mean_length"].append(lengths.mean())

            print(
                f"Ep {episode:4d} | "
                f"greedy={lengths.mean():6.1f} | "
                f"eps={epsilon_at(global_step, cfg):.3f} | "
                f"lr={lr_at(global_step, cfg):.2e}"
            )

            plot_progress(history, eval_history)

            if cfg["render_eval"]:
                length, frames = greedy_episode(
                    q_net,
                    cfg,
                    cfg["seed"] + 200_000 + episode,
                    render=True
                )

                print(f"Rendered greedy episode: {length} steps")
                show_episode(
                    frames,
                    frame_skip=cfg["frame_skip"]
                )

    env.close()

    return q_net, history, eval_history

## Ejecutar


In [ ]:
# Pueden correr de nuevo con cofig distinta para experimentar
#CONFIG =
model, history, eval_history = train(CONFIG)

## Evaluación final

In [ ]:
final_lengths = evaluate(
    model,
    CONFIG,
    CONFIG["final_eval_episodes"],
    seed_offset=300_000
)

print(
    f"Final greedy eval | "
    f"mean={final_lengths.mean():.1f} | "
    f"std={final_lengths.std():.1f} | "
    f"min={final_lengths.min()} | "
    f"max={final_lengths.max()}"
)

final_length, final_frames = greedy_episode(
    model,
    CONFIG,
    CONFIG["seed"] + 400_000,
    render=True
)

print(f"Final rendered episode: {final_length} steps")

show_episode(
    final_frames,
    frame_skip=CONFIG["frame_skip"]
)

plot_progress(history, eval_history)